# Inference: any model, any prompt (OpenRouter)

One notebook for every experiment on OpenRouter. You choose a **prompt** (a `.txt` file on Drive) and a **model** (an OpenRouter slug); the notebook runs the 12 databases for each run and saves the predictions in the layout the evaluation notebook expects.

**Folders on Drive** (inside `Repository/`):
- `prompts/<PROMPT_NAME>.txt`: one file per prompt variant, plain text, only the instructions (the JSON schema and the sample documents are appended by the notebook, as in the Qwen baseline).
- `predictions/<MODEL_TAG>_<PROMPT_NAME>/Run 1, Run 2, ...`: written by this notebook. A copy of the prompt used is saved there as `prompt_used.txt`.

**Kept equal to the Qwen baseline:** input format, output schema (enforced as JSON schema), temperature 0.3, reasoning off, file naming.

**Robust to interruptions:** files that already exist are skipped, so after a disconnection or a quota stop you rerun cell 5 and it continues.

**Run order:** cell 1 (setup), cell 2 (choose prompt and model), cell 3 (definitions), cell 4 (one database, to test), cell 5 (all runs), cell 6 (check and cost).

In [1]:
# 1. Setup: install the client, read the API key, mount Drive. Once per runtime.
#    The key is read from Colab Secrets: add a secret named OPENROUTER_API_KEY and enable notebook access.

!pip install -q -U openai pydantic json-repair

from openai import OpenAI
from google.colab import userdata, drive

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
drive.mount("/content/drive")

Mounted at /content/drive


In [28]:
# 2. Experiment: choose the prompt and the model. This is the only cell to edit between experiments.

import os, glob

PROMPT_NAME = "zero_shot"            # reads prompts/<PROMPT_NAME>.txt
MODEL = "anthropic/claude-opus-5.5"          # OpenRouter slug, as written on the model page
RUNS = [1, 2, 3]                    # run k uses seed k
TEMPERATURE = 0.3                   # same as the Qwen baseline
MAX_TOKENS = 8000                   # max output tokens; stops runs where the model loops
REASONING = "low"                  # "none" = no thinking, as the baseline. Models with mandatory reasoning reject it: use "low"
SEND_SEED = False                   # set False for models whose providers ignore the seed (e.g. Anthropic), or no provider is found
SEND_TEMPERATURE = False            # set False for models whose providers ignore the temperature (e.g. Anthropic), or no provider is found
SLEEP_S = 2                         # pause between calls

ROOT = "/content/drive/MyDrive/Applied Data Science Project/Repository"
PROMPTS_DIR = f"{ROOT}/prompts"
MODEL_TAG = MODEL.split("/")[-1].replace(":", "-")
PRED_DIR = f"{ROOT}/predictions/{MODEL_TAG}_{PROMPT_NAME}"

os.makedirs(PROMPTS_DIR, exist_ok=True)

print("Available prompts:", sorted(os.path.basename(p)[:-4] for p in glob.glob(f"{PROMPTS_DIR}/*.txt")))
PROMPT_INSTRUCTIONS = open(f"{PROMPTS_DIR}/{PROMPT_NAME}.txt", encoding="utf-8").read().strip()
print(f"\nPrompt '{PROMPT_NAME}': {len(PROMPT_INSTRUCTIONS)} characters, first lines:")
print("\n".join(PROMPT_INSTRUCTIONS.splitlines()[:4]), "...")


os.makedirs(PRED_DIR, exist_ok=True)
used = f"{PRED_DIR}/prompt_used.txt"
if os.path.exists(used) and open(used, encoding="utf-8").read().strip() != PROMPT_INSTRUCTIONS:
    raise SystemExit(f"{used} differs from prompts/{PROMPT_NAME}.txt. Use a new PROMPT_NAME for a changed prompt.")
open(used, "w", encoding="utf-8").write(PROMPT_INSTRUCTIONS)

print(f"\nModel: {MODEL}\nPredictions: {PRED_DIR}")
print(f"For the evaluation notebook: MODEL = '{MODEL_TAG}', RUN_DIRS = '{PRED_DIR}/Run k'")

Available prompts: ['zero_shot', 'zero_shot_definitions']

Prompt 'zero_shot': 1914 characters, first lines:
You are given a MongoDB database, described by its JSON schema and its sample documents.

Analyze it and return 4 separate and independent elements:
 ...

Model: anthropic/claude-opus-5.5
Predictions: /content/drive/MyDrive/Applied Data Science Project/Repository/predictions/claude-opus-5.5_zero_shot
For the evaluation notebook: MODEL = 'claude-opus-5.5', RUN_DIRS = '/content/drive/MyDrive/Applied Data Science Project/Repository/predictions/claude-opus-5.5_zero_shot/Run k'


In [29]:
# 3. Definitions: dataset folders, output schema, API call, run function.

import json, time, random
from pydantic import BaseModel, ConfigDict
from openai import APIStatusError, APIConnectionError, APITimeoutError
from json_repair import repair_json

FOLDERS = {level: glob.glob(f"/content/drive/**/{name}", recursive=True)[0]
           for level, name in [("easy", "Simple Cases"), ("medium", "Medium Cases"), ("hard", "Hard Cases")]}
DATABASES = [("easy", [1, 2, 3, 4]), ("medium", [5, 6, 7, 8]), ("hard", [9, 10, 11, 12])]


# Output schema: same fields as the Qwen baseline. extra="forbid" adds additionalProperties: false, needed by strict mode.
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")

class PatternScores(Strict):
    Embedding: float
    Referencing: float
    ExtendedReference: float
    Subset: float
    Array: float
    Bucket: float

class CollectionFields(Strict):
    collection: str
    fields: list[str]

class Relationship(Strict):
    from_field: str
    to_collection: str

class PatternPrediction(Strict):
    field: str
    evidence: str
    scores: PatternScores

class Output(Strict):
    collections: list[str]
    fields: list[CollectionFields]
    relationships: list[Relationship]
    patterns: list[PatternPrediction]

RESPONSE_FORMAT = {"type": "json_schema",
                   "json_schema": {"name": "mongodb_analysis", "strict": True, "schema": Output.model_json_schema()}}


def call_model(prompt, seed, max_retries=4):
    """One request. Retries only on temporary errors (429, 408, 5xx, network), with backoff and jitter."""
    wait = 20
    for attempt in range(max_retries):
        try:
            optional = {"seed": seed} if SEND_SEED else {}
            if SEND_TEMPERATURE:
                optional["temperature"] = TEMPERATURE
            return client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_TOKENS,
                **optional,
                response_format=RESPONSE_FORMAT,
                extra_body={
                    "reasoning": {"effort": REASONING},
                    "provider": {"require_parameters": True},   # only providers that support every parameter sent (JSON schema, seed)
                    "usage": {"include": True},                  # returns the cost of the call
                },
            )
        except (APIStatusError, APIConnectionError, APITimeoutError) as e:
            code = getattr(e, "status_code", None)
            transient = code is None or code in (408, 429) or code >= 500
            if transient and attempt < max_retries - 1:
                delay = wait + random.uniform(0, wait * 0.2)
                print(f"  error {code}, retry in {delay:.0f} s")
                time.sleep(delay)
                wait = min(wait * 2, 300)
            else:
                raise


def log_usage(level, n, run_id, usage, finish, seconds):
    path = f"{PRED_DIR}/token_usage.csv"
    new = not os.path.exists(path)
    with open(path, "a") as f:
        if new:
            f.write("db,run,input_tokens,output_tokens,cost_usd,finish,seconds\n")
        cost = getattr(usage, "cost", None)
        f.write(f"{level}{n},{run_id},{usage.prompt_tokens},{usage.completion_tokens},{cost if cost is not None else ''},{finish},{seconds:.0f}\n")


def run(level, n, run_id):
    """One database, one run. Skips it if the prediction already exists."""
    out_dir = f"{PRED_DIR}/Run {run_id}"
    name = f"{out_dir}/{level}{n}_{MODEL_TAG}"
    if os.path.exists(f"{name}.json"):
        print(f"{level}{n} Run {run_id}: already done, skipped")
        return None

    base = FOLDERS[level]
    samples = json.load(open(f"{base}/Sample Documents/{level}{n}_sample_documents.json"))
    schema = json.load(open(f"{base}/JSON Schema/{level}{n}_json_schema.json"))
    prompt = f"""{PROMPT_INSTRUCTIONS}

JSON schema:
{json.dumps(schema)}

Sample documents:
{json.dumps(samples)}"""

    t0 = time.time()
    resp = call_model(prompt, seed=run_id)
    choice = resp.choices[0]
    finish = choice.finish_reason
    text = choice.message.content or ""
    print(f"{level}{n} Run {run_id}: input={resp.usage.prompt_tokens}, output={resp.usage.completion_tokens}, "
          f"stop={finish}, {time.time() - t0:.0f} s")
    os.makedirs(out_dir, exist_ok=True)
    log_usage(level, n, run_id, resp.usage, finish, time.time() - t0)

    # Output cut by MAX_TOKENS: repair the JSON and keep complete entries, as in the Qwen baseline.
    truncated = finish == "length"
    if truncated:
        open(f"{name}_RAW.txt", "w").write(text)
        data = json.loads(repair_json(text)) if text.strip() else {}
        keys = set(PatternScores.model_fields)
        data["patterns"] = [p for p in data.get("patterns", [])
                            if isinstance(p, dict) and {"field", "evidence", "scores"} <= set(p)
                            and isinstance(p["scores"], dict) and keys <= set(p["scores"])]
        for k in ("collections", "fields", "relationships"):
            data.setdefault(k, [])
        text = json.dumps(data)
        print("  output truncated: JSON repaired, raw text saved as _RAW.txt")

    try:
        result = Output.model_validate_json(text)
    except Exception as e:
        open(f"{name}_RAW.txt", "w").write(text)
        print("  invalid JSON, raw text saved as _RAW.txt, not saved as prediction:", str(e)[:200])
        return None

    out = result.model_dump()
    out["truncated"] = truncated
    tmp = f"{name}.json.tmp"
    with open(tmp, "w") as f:
        json.dump(out, f, indent=2)
    os.replace(tmp, f"{name}.json")      # never leaves a half-written .json
    print(f"  {len(result.patterns)} patterns, saved")
    return result

In [14]:
# 4. One database, first run: check that model, prompt and schema work before the full loop.
result = run("medium", 6, RUNS[0])

medium6 Run 1: input=9163, output=2143, stop=stop, 14 s
  8 patterns, saved


In [30]:
# 5. All databases, all runs. Existing files are skipped: rerun this cell to continue after a stop.
for run_id in RUNS:
    for level, nums in DATABASES:
        for n in nums:
            try:
                if run(level, n, run_id) is not None:
                    time.sleep(SLEEP_S)
            except FileNotFoundError as e:
                print(f"{level}{n}: dataset file not found, skipped ({e.filename})")
            except APIStatusError as e:
                print(f"{level}{n} Run {run_id}: error {e.status_code}, stopping. {str(e)[:200]}")
                raise SystemExit

easy1 Run 1: input=5897, output=638, stop=stop, 10 s
  4 patterns, saved
easy2 Run 1: input=6592, output=976, stop=stop, 12 s
  5 patterns, saved
easy3 Run 1: input=6249, output=850, stop=stop, 11 s
  5 patterns, saved
easy4 Run 1: input=6160, output=603, stop=stop, 8 s
  3 patterns, saved
medium5 Run 1: input=10670, output=1708, stop=stop, 19 s
  12 patterns, saved
medium6 Run 1: input=12191, output=1880, stop=stop, 21 s
  12 patterns, saved
medium7 Run 1: input=11264, output=1323, stop=stop, 15 s
  7 patterns, saved
medium8 Run 1: input=13753, output=1803, stop=stop, 20 s
  9 patterns, saved
hard9 Run 1: input=14559, output=2481, stop=stop, 27 s
  17 patterns, saved
hard10 Run 1: input=15516, output=2335, stop=stop, 26 s
  15 patterns, saved
hard11 Run 1: input=18192, output=2719, stop=stop, 30 s
  16 patterns, saved
hard12 Run 1: input=14900, output=2873, stop=stop, 31 s
  20 patterns, saved
easy1 Run 2: input=5897, output=650, stop=stop, 9 s
  4 patterns, saved
easy2 Run 2: input=6